In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

API_KEY = '3eb8a421b534a4c6993e631c57ff8357'

df = pd.read_excel('DI-Fittings_Industry_US (1).xlsx')

def get_revenue(company):
    search_query = f"{company} revenue"
    payload = {
        'api_key': API_KEY,
        'url': f'https://www.google.com/search?q={search_query.replace(" ", "+")}'
    }
    try:
        response = requests.get('https://api.scraperapi.com/', params=payload)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Try to get revenue from Knowledge Panel
        spans = soup.find_all('span')
        for span in spans:
            if 'Revenue' in span.text:
                parent = span.find_parent()
                if parent:
                    possible_values = parent.find_all('span')
                    for val in possible_values:
                        if '$' in val.text or 'USD' in val.text:
                            return val.text.strip()

        # Fallback: Search any div with revenue and $/USD
        for div in soup.find_all('div'):
            if 'revenue' in div.text.lower() and ('$' in div.text or 'USD' in div.text):
                return div.text.strip()

        return "Not Found"
    except:
        return "Error"

revenues = []
for company in df['Major Buyers of DI Fittings']:
    if company != 'Not Found':
        revenue = get_revenue(company)
        print(f"{company}: {revenue}")
        revenues.append(revenue)
        time.sleep(5)
    else:
        revenues.append('Not Found')

df['Revenue'] = revenues
df.to_excel('out.xlsx', index=False)

print("Revenue extraction complete. Check the output file.")


New York City DEP, Philadelphia Water Dept, Boston Water & Sewer Commission: Not Found
Port Authority of NY & NJ: Not Found
Chicago Dept of Water, Cleveland Water, Milwaukee Water Utility: Filters and topicsAllNewsImagesShort videosForumsVideosShoppingMoreAbout 3,84,000 results (0.46 seconds)   Search ResultsDid you mean: Chicago Department of Water, Cleveland Water, Milwaukee Water Utility revenue AI overviewAn AI Overview is not available for this searchCan't generate an AI overview right now. Try again later.Searching  Top 10 cities with the most lead pipesEnvironmental Defense Fundhttps://www.edf.org › top-10-cities-most-lead-pipesEnvironmental Defense Fundhttps://www.edf.org › top-10-cities-most-lead-pipes23 Apr 2024 — Chicago Department of Water Management reported 387,095 LSLs in 2021 ... Cleveland Water reported 185,409 LSLs in 2021, representing ...Water and Sewer Rate StudyCHPC New Yorkhttps://chpcny.org › uploads › 2010/01 › Power...CHPC New Yorkhttps://chpcny.org › uploads 

In [2]:
import pandas as pd
import re

# Load your file
df = pd.read_excel('Ensun_Ductile_Iron_Pipe_Distributors_USA_With_Revenue.xlsx')

# Function to extract and clean revenue as Million USD format
def extract_clean_revenue(text):
    if isinstance(text, str):
        match = re.search(r'(\$)?([\d,.]+)\s*(million|billion|M|B)', text, re.IGNORECASE)
        if match:
            number = match.group(2).replace(',', '')
            unit = match.group(3).lower()

            try:
                number = float(number)
                if unit in ['billion', 'b']:
                    number *= 1000  # Convert billion to million
                return f"{round(number, 2)}M"
            except:
                return None
    return None

# Apply to Revenue column
df['Clean Revenue (M)'] = df['Revenue'].apply(extract_clean_revenue)

# Save to new file
df.to_excel('Ensun_Ductile_Iron_Pipe_Distributors_USA_With_Clean_Revenue.xlsx', index=False)

print("Revenue data cleaned successfully. Check the 'Clean Revenue (M)' column in the new file.")


Revenue data cleaned successfully. Check the 'Clean Revenue (M)' column in the new file.
